In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression

# Load dataset

df = pd.read_csv("spambase.data", header=None)

X = df.iloc[:, :-1].to_numpy(dtype=float)
y = df.iloc[:, -1].to_numpy(dtype=float)

# Train / Test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Scale features

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# add bias term
Xtr = np.c_[np.ones((X_train.shape[0], 1)), X_train]
Xte = np.c_[np.ones((X_test.shape[0], 1)), X_test]

# Sigmoid function

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

# Cross entropy loss

def cross_entropy(X, y, theta):
    N = len(y)
    probs = sigmoid(X @ theta)

    eps = 1e-12
    probs = np.clip(probs, eps, 1 - eps)

    loss = -(1/N) * np.sum(
        y*np.log(probs) + (1-y)*np.log(1-probs)
    )

    return loss

# Gradient Descent

def gradient_descent(X, y, alpha, iters):

    N, d = X.shape
    theta = np.zeros(d)

    losses = {}

    for i in range(1, iters+1):

        probs = sigmoid(X @ theta)

        grad = (1/N) * (X.T @ (probs - y))

        theta = theta - alpha * grad

        if i in [10,50,100]:
            losses[i] = cross_entropy(X, y, theta)

    return theta, losses


# Prediction functions

def predict(X, theta, threshold=0.5):

    probs = sigmoid(X @ theta)

    return (probs >= threshold).astype(int)


# Evaluation metrics

def evaluate(X, y, theta):

    preds = predict(X, theta)

    acc = accuracy_score(y, preds)
    prec = precision_score(y, preds)
    rec = recall_score(y, preds)
    f1 = f1_score(y, preds)

    return acc, prec, rec, f1


# Run Gradient Descent

alphas = [0.01, 0.1, 0.5]

loss_rows = []
metric_rows = []

for a in alphas:

    theta, losses = gradient_descent(Xtr, y_train, a, 100)

    loss_rows.append({
        "alpha":a,
        "loss_10":losses[10],
        "loss_50":losses[50],
        "loss_100":losses[100]
    })

    train_metrics = evaluate(Xtr, y_train, theta)
    test_metrics = evaluate(Xte, y_test, theta)

    metric_rows.append({
        "alpha":a,
        "Train Accuracy":train_metrics[0],
        "Train Precision":train_metrics[1],
        "Train Recall":train_metrics[2],
        "Train F1":train_metrics[3],
        "Test Accuracy":test_metrics[0],
        "Test Precision":test_metrics[1],
        "Test Recall":test_metrics[2],
        "Test F1":test_metrics[3]
    })

loss_table = pd.DataFrame(loss_rows)
metrics_table = pd.DataFrame(metric_rows)

print("Cross Entropy Loss Table")
print(loss_table)

print("\nGradient Descent Metrics")
print(metrics_table)

# Compare with sklearn

model = LogisticRegression(max_iter=5000)

model.fit(X_train, y_train)

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

print("\nSklearn Logistic Regression")

print("Train Accuracy:",accuracy_score(y_train,train_pred))
print("Test Accuracy:",accuracy_score(y_test,test_pred))

print("Precision:",precision_score(y_test,test_pred))
print("Recall:",recall_score(y_test,test_pred))
print("F1:",f1_score(y_test,test_pred))

Cross Entropy Loss Table
   alpha   loss_10   loss_50  loss_100
0   0.01  0.651167  0.541620  0.468734
1   0.10  0.465377  0.324124  0.289078
2   0.50  0.320251  0.258894  0.243778

Gradient Descent Metrics
   alpha  Train Accuracy  Train Precision  Train Recall  Train F1  \
0   0.01        0.897681         0.873389      0.860987  0.867143   
1   0.10        0.907826         0.911290      0.844544  0.876649   
2   0.50        0.918261         0.923756      0.860239  0.890867   

   Test Accuracy  Test Precision  Test Recall   Test F1  
0       0.903562        0.900881     0.861053  0.880517  
1       0.906169        0.923788     0.842105  0.881057  
2       0.915725        0.939535     0.850526  0.892818  

Sklearn Logistic Regression
Train Accuracy: 0.9257971014492754
Test Accuracy: 0.9226759339704604
Precision: 0.9406392694063926
Recall: 0.8673684210526316
F1: 0.9025191675794085
